# Chapter 7: Model Selection & Tuning
## 7.1 GridSearchCV · 7.2 RandomizedSearchCV · 7.3 Pipelines

**Problem:** Predict crop yield (kg/hectare) in Tamil Nadu based on rainfall, temperature, soil type, fertilizer usage, and crop type.

---

## The Core Problem This Chapter Solves

For 6 chapters, every time we wrote something like `RandomForestRegressor(max_depth=6, n_estimators=200)` — we **guessed** those numbers.

We didn't calculate them. We didn't prove they were optimal. We just typed numbers that felt reasonable.

**The problem:** The same algorithm on the same data with different hyperparameters can give very different results. `max_depth=6` might give RMSE of 120. `max_depth=9` might give RMSE of 85. You'll never know if you only try one value.

**The solution:** Systematically try many combinations and let the computer find the best one. That's Chapter 7.

---

## Hyperparameter vs Parameter — Critical Distinction

| | Parameter | Hyperparameter |
|---|---|---|
| **What is it?** | Learned by the model during training | Set by YOU before training |
| **Who finds it?** | The algorithm (math/optimisation) | You (guessing, or Chapter 7) |
| **Examples** | Weights w in Linear Regression, leaf values in Decision Tree | max_depth, n_estimators, learning_rate, alpha |
| **When is it set?** | After .fit() completes | Before .fit() is called |

When you call `model.fit(X, y)` — the model finds its own **parameters** automatically.
But **hyperparameters** like `max_depth` — you had to set those yourself. Chapter 7 automates that.

## Setup — Imports and Synthetic Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, randint

np.random.seed(42)

In [ ]:
# --- Synthetic Tamil Nadu Crop Yield Dataset ---
# Features that realistically affect crop yield:
# rainfall (mm), temperature (°C), fertilizer (kg/hectare),
# soil_quality (1-10 score), crop_type (encoded), irrigation (0/1)

n = 1000

rainfall        = np.random.uniform(400, 1200, n)       # mm per season
temperature     = np.random.uniform(22, 38, n)          # degrees Celsius
fertilizer      = np.random.uniform(50, 300, n)         # kg per hectare
soil_quality    = np.random.uniform(3, 10, n)           # quality score
irrigation      = np.random.randint(0, 2, n)            # 0=rain-fed, 1=irrigated
crop_type       = np.random.choice([0, 1, 2, 3], n)    # rice, wheat, sugarcane, cotton

# Yield is a function of all features + some noise
# More rainfall, fertilizer, soil quality, irrigation → higher yield
# Very high temperature → reduces yield (non-linear)
yield_kg = (
    2.5 * rainfall
    + 15 * fertilizer
    + 80 * soil_quality
    + 300 * irrigation
    - 20 * (temperature - 28)**2      # optimal temp ~28°C, penalise deviation
    + 100 * crop_type
    + np.random.normal(0, 200, n)     # real-world noise
)

df = pd.DataFrame({
    'rainfall':     rainfall,
    'temperature':  temperature,
    'fertilizer':   fertilizer,
    'soil_quality': soil_quality,
    'irrigation':   irrigation,
    'crop_type':    crop_type,
    'yield_kg':     yield_kg
})

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# --- Train/Test Split ---
# Standard 80/20 split
# X = all features, y = target (yield)

X = df.drop('yield_kg', axis=1)
y = df['yield_kg']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

---

## Baseline — Default RandomForest (No Tuning)

Before tuning, always establish a baseline. This tells us how much tuning actually helps.

Default hyperparameters are sklearn's best guesses — reasonable starting points, but not optimised for your specific data.

In [ ]:
# --- Baseline Model — Default Hyperparameters ---
# We're NOT guessing hyperparameters here — we're using sklearn defaults
# Purpose: establish a benchmark to measure improvement from tuning

baseline = RandomForestRegressor(random_state=42)
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

print(f"Baseline RMSE (default hyperparameters): ₹{rmse_baseline:.2f} kg/hectare")
print(f"Default hyperparameters used:")
print(f"  n_estimators : {baseline.n_estimators}")
print(f"  max_depth    : {baseline.max_depth}  (None = grow until pure leaves)")
print(f"  min_samples_split: {baseline.min_samples_split}")

---

## Section 7.1 — GridSearchCV

### Analogy

Imagine a chef trying to perfect a biryani recipe. She has 3 variables to experiment with:
- Rice soaking time: 20 min, 30 min, 40 min
- Spice level: low, medium, high
- Dum cooking time: 15 min, 20 min, 25 min

GridSearchCV is like that chef **systematically cooking every possible combination** — 3×3×3 = 27 biryanis — getting 5 people to taste each one, averaging their ratings, and declaring the winner.

She doesn't guess. She tries everything.

### Why "Grid"?

If you lay out 2 hyperparameters on x and y axes — you get a grid. Every cell in that grid is one combination. GridSearchCV visits every cell.

```
              n_estimators
              100   200   300
max_depth 3 | [A]  [B]  [C]
          5 | [D]  [E]  [F]
          7 | [G]  [H]  [I]
```

Each cell = one model fit (×5 folds = 5 actual fits per cell).

### Why "CV" (Cross-Validation) is built in?

If you test all 27 combinations on the **same** validation set and pick the best — you risk picking a combination that got **lucky** on that particular split. It looks great on your validation set but fails on new data. This is called **overfitting to the validation set**.

Solution: evaluate each combination across **5 different splits** and average the scores. One lucky split can't fool you.

### Full GridSearchCV Flow

```
For each combination in param_grid:
    Run 5-fold CV:
        Fold 1: train on folds 2,3,4,5 → validate on fold 1 → score
        Fold 2: train on folds 1,3,4,5 → validate on fold 2 → score
        Fold 3: train on folds 1,2,4,5 → validate on fold 3 → score
        Fold 4: train on folds 1,2,3,5 → validate on fold 4 → score
        Fold 5: train on folds 1,2,3,4 → validate on fold 5 → score
        Average 5 scores → this combination's final score

Pick combination with best average score.
Refit that combination on ALL training data (100%, no holdout).
That refitted model is what you use for predictions.
```

**Why refit on all training data?**
During CV, every model was trained on only 4 folds (80% of training data). After we know the best combination, we retrain it on 100% of training data — more data = better model.

In [ ]:
# --- GridSearchCV ---

# Step 1: Define the model (no hyperparameters — GridSearch will set them)
rf_model = RandomForestRegressor(random_state=42)

# Step 2: Define the search grid
# This is a dictionary: hyperparameter name → list of values to try
# Total combinations: 3 × 3 × 3 = 27
# Total fits: 27 combinations × 5 folds = 135 model fits
param_grid = {
    'n_estimators':     [100, 200, 300],   # number of trees in the forest
    'max_depth':        [5, 10, None],     # None = fully grown trees
    'min_samples_split': [2, 5, 10]        # minimum samples required to split a node
}

# Step 3: Create GridSearchCV object
grid_search = GridSearchCV(
    estimator  = rf_model,                      # the model to tune
    param_grid = param_grid,                    # the grid of hyperparameters to search
    cv         = 5,                             # 5-fold cross-validation for each combination
    scoring    = 'neg_mean_squared_error',       # metric to optimise
                                                # WHY neg_: GridSearchCV always MAXIMISES the score
                                                # MSE should be LOW for good models
                                                # Flip sign → neg MSE should be HIGH (close to 0)
                                                # so GridSearch correctly picks the best model
    n_jobs     = -1,                            # use ALL CPU cores in parallel — much faster
                                                # -1 is sklearn shorthand for "use everything"
    verbose    = 1                              # print progress while running
)

# Step 4: Run the search
# This runs 135 model fits internally — all combinations × all folds
# Then automatically refits the best combination on all of X_train
grid_search.fit(X_train, y_train)

print("\nGridSearchCV complete!")

In [ ]:
# --- Inspect Results ---
# GridSearchCV gives 3 key attributes after fitting:

# 1. best_params_ → the winning combination of hyperparameters
print("Best Hyperparameters:")
print(grid_search.best_params_)

# 2. best_score_ → average CV score of the winning combination
# Note: this is NEGATIVE MSE — flip sign and sqrt to get RMSE
best_cv_rmse = np.sqrt(-grid_search.best_score_)
print(f"\nBest CV RMSE: {best_cv_rmse:.2f} kg/hectare")

# 3. best_estimator_ → the actual refitted model, ready for predictions
# This model was retrained on ALL of X_train with the best hyperparameters
best_rf = grid_search.best_estimator_

# Evaluate on test set (data the model has NEVER seen)
y_pred_grid = best_rf.predict(X_test)
rmse_grid = np.sqrt(mean_squared_error(y_test, y_pred_grid))

print(f"\nTest RMSE after GridSearchCV: {rmse_grid:.2f} kg/hectare")
print(f"Baseline RMSE (no tuning):    {rmse_baseline:.2f} kg/hectare")
print(f"Improvement:                  {rmse_baseline - rmse_grid:.2f} kg/hectare")

In [ ]:
# --- Visualise the Search Results ---
# Look at all 27 combinations and their CV scores
# Helps understand the hyperparameter landscape

results = pd.DataFrame(grid_search.cv_results_)

# Convert neg MSE to RMSE for readability
results['mean_test_rmse'] = np.sqrt(-results['mean_test_score'])

# Sort by RMSE — best first
results_sorted = results[['param_n_estimators', 'param_max_depth',
                           'param_min_samples_split', 'mean_test_rmse']]\
                          .sort_values('mean_test_rmse')

print("Top 5 combinations:")
print(results_sorted.head())
print("\nBottom 5 combinations:")
print(results_sorted.tail())

### When GridSearchCV Fails You

GridSearchCV tries **every** combination. That's great when you have a small grid. But it explodes fast:

| Hyperparameters | Values each | Combinations | Fits (5-fold) |
|---|---|---|---|
| 3 | 3 | 27 | 135 |
| 5 | 5 | 3,125 | 15,625 |
| 8 | 10 | 100,000,000 | 500,000,000 |

XGBoost has 8+ hyperparameters. Exhaustive search becomes computationally impossible.

That's what RandomizedSearchCV solves.

---

## Section 7.2 — RandomizedSearchCV

### Analogy

Same chef. Same biryani problem. But now she has **10 variables** with 10 values each — 10 billion combinations. She can't cook all of them in her lifetime.

So instead she randomly picks 50 combinations, cooks those, and picks the best among them.

She might miss the **absolute** best biryani in the universe. But she'll find a very good one — in a tiny fraction of the time.

### Two Advantages Over GridSearchCV

**1. Speed:** Instead of all combinations, you choose how many to try (`n_iter`). 50 fits instead of 500 million.

**2. Continuous Distributions:** GridSearchCV takes fixed lists `[0.01, 0.1, 0.3]`. What if the true optimal is `0.047`? Your list misses it.

RandomizedSearchCV accepts **probability distributions** — e.g., `uniform(0.01, 0.3)` means "pick any value between 0.01 and 0.3". It can find 0.047 — your fixed list never could.

### The Tradeoff

RandomizedSearchCV is **not guaranteed** to find the absolute best combination. It might miss it. But in practice, a well-chosen random sample finds a combination very close to optimal — in a fraction of the compute time.

In [ ]:
# --- RandomizedSearchCV ---

rf_model2 = RandomForestRegressor(random_state=42)

# param_distributions: like param_grid but can use scipy distributions
# randint(a, b) → random integers from a to b-1
# uniform(loc, scale) → random floats from loc to loc+scale
param_distributions = {
    'n_estimators':      randint(50, 400),      # any integer from 50 to 399
    'max_depth':         [5, 10, 15, None],     # fixed list still works too
    'min_samples_split': randint(2, 20),        # any integer from 2 to 19
    'min_samples_leaf':  randint(1, 10),        # min samples in a leaf node
    'max_features':      uniform(0.3, 0.7)      # fraction of features to consider at each split
                                                # any float from 0.3 to 1.0
}
# Total possible combinations: essentially infinite (continuous distributions)
# We'll only try n_iter=50 of them

random_search = RandomizedSearchCV(
    estimator           = rf_model2,
    param_distributions = param_distributions,
    n_iter              = 50,                   # try 50 random combinations (not all of them)
    cv                  = 5,                    # same 5-fold CV as GridSearchCV
    scoring             = 'neg_mean_squared_error',
    n_jobs              = -1,
    random_state        = 42,                   # for reproducibility — same 50 combinations each run
    verbose             = 1
)

random_search.fit(X_train, y_train)
print("\nRandomizedSearchCV complete!")

In [ ]:
# --- Inspect RandomizedSearchCV Results ---

print("Best Hyperparameters:")
print(random_search.best_params_)

best_cv_rmse_rand = np.sqrt(-random_search.best_score_)
print(f"\nBest CV RMSE: {best_cv_rmse_rand:.2f} kg/hectare")

best_rf_rand = random_search.best_estimator_

y_pred_rand = best_rf_rand.predict(X_test)
rmse_rand = np.sqrt(mean_squared_error(y_test, y_pred_rand))

print(f"\n--- Comparison ---")
print(f"Baseline RMSE (default):       {rmse_baseline:.2f}")
print(f"GridSearchCV RMSE (27 combos): {rmse_grid:.2f}")
print(f"RandomizedSearchCV RMSE (50):  {rmse_rand:.2f}")

---

## Section 7.3 — Pipelines

### The Data Leakage Problem

You've been writing this pattern for 6 chapters:

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # scaler sees ALL of X_train
X_test_scaled  = scaler.transform(X_test)

grid_search.fit(X_train_scaled, y_train)          # GridSearch splits X_train_scaled into folds
```

**The problem:** When GridSearchCV takes fold 5 as the validation fold — that fold's data was already seen by the scaler during `fit_transform(X_train)`. The scaler learned the mean and std of fold 5 and used it to scale everything — including the training folds.

This means fold 5 is **not truly unseen**. Its statistics leaked into the scaling step. Your CV scores are slightly optimistic — they look better than real-world performance.

This is **data leakage** — information from validation data contaminating the training process.

### The Fix — Pipeline

A Pipeline chains preprocessing and model into **one object**. When GridSearchCV splits into folds, the Pipeline handles each fold correctly:

```
For each fold:
    scaler.fit_transform(training_folds_only)  ← scaler NEVER sees validation fold
    scaler.transform(validation_fold)          ← validation fold scaled using training stats only
    model.fit(scaled_training_folds)
    model.predict(scaled_validation_fold)
```

The validation fold's statistics never touch the scaler. True unseen data. True CV score.

### Pipeline Syntax

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),        # step name → object
    ('model',  RandomForestRegressor())
])
```

With GridSearch, hyperparameter names use `stepname__param` syntax:
```python
param_grid = {
    'model__max_depth': [5, 10, None]    # model__ tells GridSearch: this param belongs to 'model' step
}
```

The `__` (double underscore) is sklearn's way of saying: "go inside this step and set this parameter."

Without `model__` prefix, GridSearch doesn't know if `max_depth` belongs to the scaler or the model — since both are inside the same Pipeline.

In [ ]:
# --- Pipeline + GridSearchCV (The Correct Way) ---

# Step 1: Build the Pipeline
# Chain: StandardScaler → RandomForestRegressor
# When .fit() is called on the pipeline, it runs each step in order
# When .predict() is called, it transforms X then predicts
pipe = Pipeline([
    ('scaler', StandardScaler()),          # step 1: scale the features
    ('model',  RandomForestRegressor(random_state=42))   # step 2: train the model
])

# Step 2: Define param_grid — note the stepname__ prefix
# 'model__max_depth' means: set max_depth on the 'model' step
# This is necessary because Pipeline has multiple steps — need to specify which step
param_grid_pipe = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [5, 10, None],
    'model__min_samples_split': [2, 5, 10]
}

# Step 3: GridSearchCV with the pipeline as estimator
# Now when GridSearchCV splits into folds:
#   → Pipeline scales ONLY on training folds (scaler never sees validation fold)
#   → True data leakage prevention
grid_search_pipe = GridSearchCV(
    estimator  = pipe,                           # pipeline, not just the model
    param_grid = param_grid_pipe,
    cv         = 5,
    scoring    = 'neg_mean_squared_error',
    n_jobs     = -1,
    verbose    = 1
)

# We pass RAW X_train here — NOT pre-scaled
# The Pipeline handles scaling internally, correctly, per fold
grid_search_pipe.fit(X_train, y_train)
print("\nPipeline + GridSearchCV complete!")

In [ ]:
# --- Results ---

print("Best Hyperparameters (Pipeline):")
print(grid_search_pipe.best_params_)

best_cv_rmse_pipe = np.sqrt(-grid_search_pipe.best_score_)
print(f"\nBest CV RMSE: {best_cv_rmse_pipe:.2f} kg/hectare")

# best_estimator_ is now the full pipeline (scaler + model)
# .predict() on raw X_test — pipeline scales it internally before predicting
best_pipe = grid_search_pipe.best_estimator_
y_pred_pipe = best_pipe.predict(X_test)           # no need to scale X_test manually!
rmse_pipe = np.sqrt(mean_squared_error(y_test, y_pred_pipe))

print(f"\n--- Final Comparison ---")
print(f"Baseline RMSE (default, no tuning):  {rmse_baseline:.2f} kg/hectare")
print(f"GridSearchCV RMSE (manual scaling):  {rmse_grid:.2f} kg/hectare")
print(f"RandomizedSearchCV RMSE:             {rmse_rand:.2f} kg/hectare")
print(f"Pipeline + GridSearchCV RMSE:        {rmse_pipe:.2f} kg/hectare  ← most correct approach")

In [ ]:
# --- Visualise: Actual vs Predicted (Pipeline model) ---

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_pipe, alpha=0.4, color='steelblue', edgecolors='navy', linewidth=0.3)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Yield (kg/hectare)')
plt.ylabel('Predicted Yield (kg/hectare)')
plt.title('Actual vs Predicted — Tuned RandomForest (Pipeline)')
plt.legend()
plt.tight_layout()
plt.show()

# Points along the red line = perfect predictions
# Scatter around it = prediction error

---

## Summary Table

| Tool | How it works | When to use | Key parameter |
|---|---|---|---|
| **GridSearchCV** | Tries every combination in the grid | Small search space (< few hundred combos) | `param_grid` (dict of lists) |
| **RandomizedSearchCV** | Randomly samples n_iter combinations | Large search space, many hyperparameters | `n_iter`, `param_distributions` (can use scipy distributions) |
| **Pipeline** | Chains scaler + model into one object | Always — prevents data leakage during CV | `stepname__param` syntax in param_grid |

## Key Concepts to Remember

| Concept | What it means |
|---|---|
| `neg_mean_squared_error` | MSE with flipped sign — so GridSearch can maximise it (lower MSE = higher neg MSE) |
| `n_jobs=-1` | Use all CPU cores — runs fits in parallel, much faster |
| `best_params_` | The winning hyperparameter combination |
| `best_score_` | Average CV score of winning combination (negative MSE — flip sign) |
| `best_estimator_` | The model refitted on ALL training data with best params — use this for predictions |
| `refit` | After finding best params, GridSearch automatically retrains on 100% of training data |
| Data leakage in CV | Scaling before CV lets validation fold's stats contaminate scaler — Pipeline fixes this |
| `model__max_depth` | Double underscore syntax to set params on a specific Pipeline step |

---

## Practice Task

Use `RandomizedSearchCV` with a **Pipeline** (StandardScaler + RandomForestRegressor) on the same crop yield data. Search over:
- `n_estimators`: randint(100, 500)
- `max_depth`: [5, 10, 15, None]
- `max_features`: uniform(0.3, 0.7)

Use `n_iter=30`, `cv=5`. Print best params, best CV RMSE, and test RMSE.

In [ ]:
# YOUR CODE HERE

# Step 1: Build the Pipeline

# Step 2: Define param_distributions (remember: model__ prefix)

# Step 3: Create RandomizedSearchCV with n_iter=30, cv=5

# Step 4: Fit on X_train, y_train

# Step 5: Print best_params_, best CV RMSE, test RMSE